In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
INPUT_PATH = Path(
    "results/gta_promotion_rankings_v1.parquet"
)

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    OUTPUT_DIR /
    "gta_optimized_flyer_v1.parquet"
)

In [3]:
df = pd.read_parquet(INPUT_PATH)

print(df.shape)
print(df.columns.tolist())

display(df.head())

(105, 57)
['store_id', 'store_name', 'address', 'city', 'province', 'postal_code', 'full_address', 'latitude', 'longitude', 'universities', 'colleges', 'schools', 'offices', 'industrial', 'commercial', 'retail_landuse', 'rail_stations', 'store_name_demo', 'population_5km', 'population_density_5km', 'median_household_income_5km', 'average_household_size_5km', 'employment_rate_5km', 'n_census_tracts', 'product_id', 'product_name', 'category', 'current_price', 'regular_price', 'savings', 'promotion_type', 'product_url', 'page_number', 'snapshot_date', 'country', 'region', 'source', 'captured_at', 'discount_amount', 'discount_pct', 'is_promoted', 'promo_id', 'promo_price', 'context_id', 'discount_score', 'population_score', 'income_score', 'commercial_raw', 'commercial_score', 'weather_score', 'opportunity_score', 'store_rank', 'discount_contribution', 'population_contribution', 'commercial_contribution', 'income_contribution', 'weather_contribution']


,store_id,store_name,address,city,province,postal_code,full_address,latitude,longitude,universities,...,commercial_raw,commercial_score,weather_score,opportunity_score,store_rank,discount_contribution,population_contribution,commercial_contribution,income_contribution,weather_contribution
0,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,36.131981,4,0.000000,20.0,15.0,1.131981,0.0
1,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,50.293848,2,14.161868,20.0,15.0,1.131981,0.0
2,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,36.131981,5,0.000000,20.0,15.0,1.131981,0.0
3,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,43.717070,3,7.585090,20.0,15.0,1.131981,0.0
4,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,81.131981,1,45.000000,20.0,15.0,1.131981,0.0


# Basic QA

In [8]:
ASSUMED_MARGIN_RATE = 0.25

MAX_FLYER_SLOTS = 20

MAX_PRODUCTS_PER_CATEGORY = 5

# Price sanity

In [9]:
df["promo_price"] = pd.to_numeric(
    df["promo_price"],
    errors="coerce"
)

df["regular_price"] = pd.to_numeric(
    df["regular_price"],
    errors="coerce"
)

df["discount_pct"] = pd.to_numeric(
    df["discount_pct"],
    errors="coerce"
)

In [10]:
df = df[
    df["promo_price"].notna()
    & (df["promo_price"] > 0)
].copy()

# Margin proxy

In [11]:
df["estimated_unit_margin"] = (
    df["promo_price"]
    * ASSUMED_MARGIN_RATE
)

# Convert opportunity score into demand-response proxy


In [12]:
df["opportunity_normalized"] = (
    df["opportunity_score"]
    .clip(0, 100)
    / 100
)

In [13]:
MAX_INCREMENTAL_UNITS_PROXY = 10

df["incremental_units_proxy"] = (
    df["opportunity_normalized"]
    * MAX_INCREMENTAL_UNITS_PROXY
)

# Profit proxy

In [14]:
df["profit_proxy"] = (
    df["incremental_units_proxy"]
    * df["estimated_unit_margin"]
)

# Discount-cost penalty

In [15]:
df["discount_penalty"] = (
    df["discount_pct"]
    .fillna(0)
    .clip(0, 60)
    / 60
)

In [16]:
df["economic_score"] = (
    0.70 * df["profit_proxy"]
    +
    0.30 * df["opportunity_normalized"]
)

In [17]:
df["economic_score"] = (
    df["economic_score"]
    * (
        1
        - 0.15 * df["discount_penalty"]
    )
)

# Inspect economics

In [18]:
cols = [
    c for c in [
        "store_id",
        "product_name",
        "category",
        "promo_price",
        "discount_pct",
        "opportunity_score",
        "incremental_units_proxy",
        "estimated_unit_margin",
        "profit_proxy",
        "economic_score"
    ]
    if c in df.columns
]

display(
    df[cols]
    .sort_values(
        "economic_score",
        ascending=False
    )
    .head(30)
)

,store_id,product_name,category,promo_price,discount_pct,opportunity_score,incremental_units_proxy,estimated_unit_margin,profit_proxy,economic_score
0,3106,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,36.131981,3.613198,24.2500,87.620053,61.442433
21,1150,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,29.066087,2.906609,24.2500,70.485262,49.426882
14,1004,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,27.395502,2.739550,24.2500,66.434093,46.586051
28,3105,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,20.530295,2.053029,24.2500,49.785964,34.911766
98,3130,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,17.328743,1.732874,24.2500,42.022202,29.467527
42,3055,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,17.280318,1.728032,24.2500,41.904772,29.385182
56,1126,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,17.172037,1.717204,24.2500,41.642191,29.201049
49,1061,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,15.866870,1.586687,24.2500,38.477160,26.981613
77,1079,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,13.459188,1.345919,24.2500,32.638530,22.887349
63,3054,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,11.425506,1.142551,24.2500,27.706852,19.429073


# Flyer optimization

# Optimizer

In [19]:
def optimize_store_flyer(
    store_df,
    max_slots=20,
    max_per_category=5
):

    store_df = (
        store_df
        .sort_values(
            "economic_score",
            ascending=False
        )
        .copy()
    )

    selected = []

    category_counts = {}

    seen_products = set()

    for _, row in store_df.iterrows():

        product_key = (
            row["product_id"]
            if (
                "product_id" in row.index
                and pd.notna(row["product_id"])
            )
            else row["product_name"]
        )

        if product_key in seen_products:
            continue

        category = row.get(
            "category",
            "Unknown"
        )

        if pd.isna(category):
            category = "Unknown"

        count = category_counts.get(
            category,
            0
        )

        if count >= max_per_category:
            continue

        selected.append(row)

        seen_products.add(product_key)

        category_counts[category] = (
            count + 1
        )

        if len(selected) >= max_slots:
            break

    if not selected:
        return pd.DataFrame()

    result = pd.DataFrame(selected)

    result["optimized_rank"] = (
        range(1, len(result) + 1)
    )

    return result

# Run optimizer for every Walmart

In [20]:
optimized_frames = []

for store_id, store_df in df.groupby(
    "store_id"
):

    result = optimize_store_flyer(
        store_df,
        max_slots=MAX_FLYER_SLOTS,
        max_per_category=MAX_PRODUCTS_PER_CATEGORY
    )

    optimized_frames.append(result)

In [21]:
optimized = pd.concat(
    optimized_frames,
    ignore_index=True
)

print(optimized.shape)

(75, 64)


# Verify constraints

In [22]:
slots_per_store = (
    optimized
    .groupby("store_id")
    .size()
)

display(
    slots_per_store
    .to_frame("flyer_slots")
)

,flyer_slots
store_id,
1004,5
1061,5
1079,5
1126,5
1139,5
1150,5
1188,5
3031,5
3054,5


In [23]:
print(
    "Maximum slots:",
    slots_per_store.max()
)

Maximum slots: 5


# Category constraint

In [24]:
category_check = (
    optimized
    .groupby(
        ["store_id", "category"]
    )
    .size()
    .reset_index(name="n")
)

display(
    category_check
    .sort_values(
        "n",
        ascending=False
    )
    .head(30)
)

,store_id,category,n
0,1004,Grocery,5
1,1061,Grocery,5
2,1079,Grocery,5
3,1126,Grocery,5
4,1139,Grocery,5
5,1150,Grocery,5
6,1188,Grocery,5
7,3031,Grocery,5
8,3054,Grocery,5
9,3055,Grocery,5


In [25]:
print(
    "Maximum products/category:",
    category_check["n"].max()
)

Maximum products/category: 5


# Example optimized flyer

In [26]:
example_store = (
    optimized["store_id"]
    .iloc[0]
)

example = (
    optimized[
        optimized["store_id"]
        == example_store
    ]
    .sort_values(
        "optimized_rank"
    )
)

display(
    example[
        [
            c for c in [
                "optimized_rank",
                "store_name",
                "product_name",
                "category",
                "promo_price",
                "discount_pct",
                "opportunity_score",
                "profit_proxy",
                "economic_score"
            ]
            if c in example.columns
        ]
    ]
)

,optimized_rank,store_name,product_name,category,promo_price,discount_pct,opportunity_score,profit_proxy,economic_score
0,1,Toronto Stockyards,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,27.395502,66.434093,46.586051
1,2,Toronto Stockyards,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,22.025913,34.980592,17.367864,11.587218
2,3,Toronto Stockyards,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,40.020060,72.395502,10.823128,7.013647
3,4,Toronto Stockyards,"Red Bull Energy Drink, 250 ml (4 pack)",Grocery,10.97,NaN,27.395502,7.513216,5.341438
4,5,Toronto Stockyards,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,10.97,18.377976,27.395502,7.513216,5.096026


# Store-level summary

In [27]:
store_summary = (
    optimized
    .groupby("store_id")
    .agg(
        flyer_products=(
            "optimized_rank",
            "count"
        ),

        total_profit_proxy=(
            "profit_proxy",
            "sum"
        ),

        avg_opportunity_score=(
            "opportunity_score",
            "mean"
        ),

        avg_discount_pct=(
            "discount_pct",
            "mean"
        )
    )
    .reset_index()
)

display(store_summary)

,store_id,flyer_products,total_profit_proxy,avg_opportunity_score,avg_discount_pct
0,1004,5,109.651517,37.912520,26.807983
1,1061,5,67.923633,26.383888,26.807983
2,1079,5,59.209027,23.976206,26.807983
3,1126,5,72.647686,27.689055,26.807983
4,1139,5,48.974773,21.148673,26.807983
5,1150,5,115.698200,39.583105,26.807983
6,1188,5,46.097761,20.353809,26.807983
7,3031,5,38.171114,18.163825,26.807983
8,3054,5,51.848115,21.942524,26.807983
9,3055,5,73.039610,27.797336,26.807983


# Compare naive vs optimized

In [28]:
naive_frames = []

for store_id, group in df.groupby(
    "store_id"
):

    naive = (
        group
        .sort_values(
            "discount_pct",
            ascending=False
        )
        .drop_duplicates(
            subset=["product_name"]
        )
        .head(MAX_FLYER_SLOTS)
        .copy()
    )

    naive["selection_method"] = (
        "largest_discount"
    )

    naive_frames.append(naive)

naive = pd.concat(
    naive_frames,
    ignore_index=True
)

In [29]:
comparison = pd.DataFrame({
    "method": [
        "Largest Discount",
        "Profit-Aware Optimizer"
    ],

    "total_profit_proxy": [
        naive["profit_proxy"].sum(),
        optimized["profit_proxy"].sum()
    ],

    "avg_opportunity_score": [
        naive["opportunity_score"].mean(),
        optimized["opportunity_score"].mean()
    ]
})

display(comparison)

,method,total_profit_proxy,avg_opportunity_score
0,Largest Discount,1124.450554,26.482996
1,Profit-Aware Optimizer,1077.536345,27.464734


# Save


In [30]:
optimized.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(
    "Saved:",
    OUTPUT_PATH
)

print(
    "Rows:",
    len(optimized)
)

Saved: results/gta_optimized_flyer_v1.parquet
Rows: 75


In [31]:
summary_path = (
    OUTPUT_DIR /
    "gta_optimized_flyer_summary_v1.csv"
)

store_summary.to_csv(
    summary_path,
    index=False
)

print("Saved:", summary_path)

Saved: results/gta_optimized_flyer_summary_v1.csv


In [32]:
# Local geography
# + demographics
# + weather
# + Walmart promotions
#         ↓
# Promotion opportunity ranking
#         ↓
# economic / margin proxy
#         ↓
# business constraints
#         ↓
# optimized flyer per GTA Walmart